# PANDAS III — COMPLEMENTO CODEX

#### Nota Codex — Correspondencia con el curso

Acompaña [Pandas III: importación de datos](07_Pandas_Base_III_Importacion_De_Datos.ipynb). Las secciones 1 a 4 amplían read_csv y los problemas de importación; la 5 refuerza guardar y releer; las secciones 6 y 7 completan Excel y SQL con datos pequeños. Todos los datos se generan aquí. Requiere Pandas 2.1 o posterior y openpyxl para la sección Excel.

## Objetivos de aprendizaje

1. Traducir la estructura de un archivo a parámetros de lectura.
2. Preservar identificadores, faltantes documentados y fechas.
3. Leer solo las columnas necesarias sin confundir orden ni muestra.
4. Verificar que exportar y releer conserva lo que se necesita.

## Preparación

In [1]:
from io import StringIO
from pathlib import Path
from tempfile import TemporaryDirectory
from itertools import islice
import sqlite3
import pandas as pd
from IPython.display import display

# Carpeta temporal propia: no se modifican los datasets del curso.
temporal = TemporaryDirectory(prefix='pandas_codex_')
carpeta = Path(temporal.name)
print('Los archivos de práctica se crean en una carpeta temporal.')

Los archivos de práctica se crean en una carpeta temporal.


#### Nota Codex — Rutas, codificación y recursos

Una ruta relativa se interpreta desde Path.cwd(), no desde el nombre del notebook. `Path` permite componer rutas sin escribir separadores propios de Windows. `encoding` indica cómo se representan los caracteres; no es el separador de columnas. Usaremos UTF-8 y archivos creados aquí para evitar dependencias externas. Al final se elimina únicamente la carpeta temporal de esta sesión.

## 1. Cabecera presente, ausente y metadatos

#### Nota Codex — Concepto y uso

`header=0` consume la primera fila útil como nombres. Si además das `names`, reemplaza esos nombres. `header=None` indica que la primera fila útil ya contiene datos. `skiprows=1` omite una línea inicial del archivo.

StringIO convierte un texto en un objeto que read_csv puede leer como un archivo. Cada llamada crea uno nuevo, por lo que no dependemos de la posición de lectura de un objeto reutilizado.

In [2]:
sin_cabecera = '001;Ana;12\n002;Luis;18\n'
columnas = ['codigo', 'nombre', 'unidades']
correcta = pd.read_csv(StringIO(sin_cabecera), sep=';', header=None,
                      names=columnas, dtype={'codigo': 'string'})
equivocada = pd.read_csv(StringIO(sin_cabecera), sep=';', header=0,
                        names=columnas, dtype={'codigo': 'string'})
display(correcta)
print('Filas correctas:', len(correcta), 'Con header equivocado:', len(equivocada))
con_metadatos = 'Reporte de práctica\ncodigo;nombre;unidades\n001;Ana;12\n'
display(pd.read_csv(StringIO(con_metadatos), sep=';', skiprows=1,
                    header=0, names=columnas, dtype={'codigo': 'string'}))

Filas correctas: 2 Con header equivocado: 1


,codigo,nombre,unidades
0,001,Ana,12
1,002,Luis,18


,codigo,nombre,unidades
0,001,Ana,12


#### Nota Codex — Lectura del resultado

La lectura correcta conserva dos registros; la equivocada pierde a Ana sin lanzar error. Que read_csv termine no basta para afirmar que la importación sea correcta. En el último ejemplo sí existe una cabecera, después de una línea de metadatos.

## 2. Un pequeño contrato de lectura

#### Nota Codex — Concepto y uso

La fuente documenta: campos separados por punto y coma, importes con coma decimal y punto de miles, códigos de tres caracteres y fechas día/mes/año. También indica que SIN_DATO significa precio desconocido y que NA es un código de zona **válido**.

`dtype` protege los ceros iniciales. `keep_default_na=False` desactiva los marcadores de ausencia automáticos; `na_values` declara los de esta fuente por columna. Esto exige incluir también la cadena vacía si la fuente la usa como faltante. No estamos imputando datos.

In [3]:
texto = ('codigo;zona;precio;fecha\n'
         '001;NA;1.250,50;02/04/2026\n'
         '010;SUR;SIN_DATO;03/04/2026\n'
         '020;NORTE;35,00;04/04/2026\n')
archivo = carpeta / 'precios.csv'
archivo.write_text(texto, encoding='utf-8')
precios = pd.read_csv(
    archivo, sep=';', encoding='utf-8', decimal=',', thousands='.',
    dtype={'codigo': 'string', 'zona': 'string'},
    keep_default_na=False, na_values={'precio': ['SIN_DATO', '']},
    parse_dates=['fecha'], date_format='%d/%m/%Y'
)
display(precios)
display(precios.dtypes)
assert precios.loc[0, 'codigo'] == '001'
assert precios.loc[0, 'zona'] == 'NA'
assert precios.loc[0, 'precio'] == 1250.5
assert precios.loc[0, 'fecha'] == pd.Timestamp('2026-04-02')

,codigo,zona,precio,fecha
0,001,NA,1250.5,2026-04-02
1,010,SUR,NaN,2026-04-03
2,020,NORTE,35.0,2026-04-04


codigo    string[python]
zona      string[python]
precio           float64
fecha     datetime64[ns]
dtype: object

#### Nota Codex — Lectura del resultado

001 sigue siendo texto, NA no desaparece y 02/04 significa 2 de abril. El segundo precio queda faltante. Los assert verifican acuerdos concretos del ejemplo; si uno falla, se detiene la ejecución para investigar la lectura. El diagnóstico completo vendrá en Pandas IV.

## 3. Vista previa, columnas y número de registros

#### Nota Codex — Concepto y uso

Para inspeccionar el archivo antes de parametrizarlo, lee pocas líneas con with e islice. `usecols` reduce las columnas que se cargan y `nrows` limita registros. El orden dado en usecols no impone el orden de salida: selecciona luego las columnas en el orden deseado con loc.

No uses los primeros registros como si fueran una muestra aleatoria; el muestreo se estudia en Pandas IV.

In [4]:
with archivo.open(encoding='utf-8') as entrada:
    print(''.join(islice(entrada, 3)))
seleccion = pd.read_csv(archivo, sep=';', encoding='utf-8',
                       usecols=['fecha', 'codigo'], nrows=2,
                       dtype={'codigo': 'string'}, parse_dates=['fecha'],
                       date_format='%d/%m/%Y')
seleccion = seleccion.loc[:, ['fecha', 'codigo']]
display(seleccion)
assert seleccion.shape == (2, 2)

codigo;zona;precio;fecha
001;NA;1.250,50;02/04/2026
010;SUR;SIN_DATO;03/04/2026



,fecha,codigo
0,2026-04-02,001
1,2026-04-03,010


#### Nota Codex — Lectura del resultado

Se cargan dos registros y dos columnas, ahora con fecha antes que codigo. La selección usa la misma mecánica de Pandas II; no necesita volver a leer todo el archivo para reordenar columnas.

## 4. Campos entre comillas y errores de lectura

#### Nota Codex — Concepto y uso

Un separador dentro de un campo no divide la columna si el campo está correctamente entre comillas. read_csv interpreta esas comillas. No conviene reemplazar separadores manualmente con str.split.

Una fila con demasiados campos puede producir ParserError. En este ejemplo lo capturamos para observarlo; en datos reales investiga la causa. `on_bad_lines="skip"` podría ocultar una pérdida de registros.

In [5]:
citado = 'codigo;comentario\n001;"Entrega; por la tarde"\n'
display(pd.read_csv(StringIO(citado), sep=';', dtype={'codigo': 'string'}))
mal_formado = 'codigo;unidades\n001;2\n002;3;campo_extra\n'
try:
    pd.read_csv(StringIO(mal_formado), sep=';')
except pd.errors.ParserError:
    print('Error esperado: una fila tiene más campos que la cabecera.')

Error esperado: una fila tiene más campos que la cabecera.


,codigo,comentario
0,001,Entrega; por la tarde


#### Nota Codex — Lectura del resultado

El comentario completo permanece en una sola columna. La segunda lectura falla por estructura, no porque falte Pandas ni porque una columna necesite una conversión. Un UnicodeDecodeError, en cambio, orientaría la revisión hacia la codificación del archivo.

## 5. Exportar y releer: los tipos no viajan solos

#### Nota Codex — Concepto y uso

Un CSV guarda texto, no un esquema Pandas. Debes acordar separador, codificación, representación de fechas, faltantes e índice. Aquí codigo sigue siendo una columna, así que no exportamos el índice automático.

Después de exportar, releemos con reglas explícitas. Esta comprobación detecta pérdidas como los ceros iniciales o una columna adicional Unnamed: 0 causada por exportar un índice sin nombre.

In [6]:
destino = carpeta / 'precios_exportados.csv'
precios.to_csv(destino, sep=';', index=False, encoding='utf-8',
               na_rep='SIN_DATO', date_format='%Y-%m-%d')
recuperado = pd.read_csv(
    destino, sep=';', encoding='utf-8',
    dtype={'codigo': 'string', 'zona': 'string'},
    keep_default_na=False, na_values={'precio': ['SIN_DATO']},
    parse_dates=['fecha'], date_format='%Y-%m-%d'
)
display(recuperado)
pd.testing.assert_frame_equal(precios, recuperado)
print('Relectura verificada: valores, columnas, índice y tipos coinciden.')

Relectura verificada: valores, columnas, índice y tipos coinciden.


,codigo,zona,precio,fecha
0,001,NA,1250.5,2026-04-02
1,010,SUR,NaN,2026-04-03
2,020,NORTE,35.0,2026-04-04


#### Nota Codex — Lectura del resultado

La salida usa punto decimal y fechas año-mes-día, aunque el archivo de origen tenía otros formatos. Los parámetros de lectura deben corresponder al **archivo exportado**. La igualdad comprobada corresponde a este ejemplo y a sus tipos declarados; CSV no conserva automáticamente cualquier objeto de Python.

## 6. Excel: nombres de hojas y resultado de la lectura

#### Nota Codex — Concepto y uso

Los archivos xlsx requieren un motor como openpyxl (`%pip install openpyxl` en el entorno del notebook si falta). No es necesario instalar Microsoft Excel para este ejemplo.

`sheet_name="precios"` devuelve un DataFrame. `sheet_name=None` devuelve un diccionario que asocia cada nombre de hoja con un DataFrame. Esta diferencia importa antes de usar loc.

In [7]:
libro = carpeta / 'catalogo.xlsx'
with pd.ExcelWriter(libro, engine='openpyxl') as escritor:
    precios.to_excel(escritor, sheet_name='precios', index=False)
    pd.DataFrame({'fuente': ['Ejemplo Codex']}).to_excel(
        escritor, sheet_name='notas', index=False)
with pd.ExcelFile(libro, engine='openpyxl') as entrada:
    print('Hojas disponibles:', entrada.sheet_names)
    hoja = pd.read_excel(entrada, sheet_name='precios',
                         dtype={'codigo': 'string', 'zona': 'string'},
                         keep_default_na=False, na_values={'precio': ['']})
hojas = pd.read_excel(libro, sheet_name=None, engine='openpyxl')
print('Tipo con sheet_name=None:', type(hojas).__name__)
display(hoja)
assert hoja.loc[0, 'codigo'] == '001'
assert hoja.loc[0, 'zona'] == 'NA'
assert pd.isna(hoja.loc[1, 'precio'])

Hojas disponibles: ['precios', 'notas']
Tipo con sheet_name=None: dict


,codigo,zona,precio,fecha
0,001,NA,1250.5,2026-04-02
1,010,SUR,NaN,2026-04-03
2,020,NORTE,35.0,2026-04-04


#### Nota Codex — Lectura del resultado

El libro tiene dos hojas. hoja es una tabla y hojas es un diccionario: usa hojas['precios'] para extraer esa tabla. La lectura cuidada protege 001 y NA; la lectura genérica de todas las hojas solo ilustra la estructura del resultado, sin garantizar esos tipos. Excel no admite fechas con zona horaria; el curso III explica la decisión necesaria antes de exportarlas.

## 7. SQL mínimo y conexión autocontenida

#### Nota Codex — Concepto y uso

El curso usa SQLAlchemy para acceder a chinook. Pandas también admite una conexión sqlite3, incluida en Python, para este caso sencillo. Creamos una base solo en memoria: no se necesita servidor ni archivo previo.

La consulta selecciona dos columnas de una tabla, sin joins. El signo ? es un parámetro de SQLite; `params` entrega el valor aparte. La sintaxis de parámetros depende del conector. No hace falta construir SQL concatenando entradas de texto.

In [8]:
conexion = sqlite3.connect(':memory:')
try:
    # Datos preparados para que el ejemplo sea autónomo.
    conexion.execute('CREATE TABLE pedidos (codigo TEXT, unidades INTEGER)')
    conexion.executemany('INSERT INTO pedidos VALUES (?, ?)',
                        [('001', 2), ('010', 5), ('020', 1)])
    pedidos = pd.read_sql_query(
        'SELECT codigo, unidades FROM pedidos WHERE unidades >= ? ORDER BY codigo',
        conexion, params=(2,), index_col='codigo'
    )
    display(pedidos)
    assert pedidos.index.tolist() == ['001', '010']
finally:
    conexion.close()

,unidades
codigo,
001,2
010,5


#### Nota Codex — Lectura del resultado

Se recuperan dos filas: 001 con 2 unidades y 010 con 5. index_col convierte codigo en etiquetas, como en read_csv. finally cierra la conexión aunque falle la consulta. Las uniones entre tablas se dejan a Pandas VI.

## Cierre

#### Nota Codex — Archivos temporales

La siguiente celda elimina únicamente los archivos de práctica generados en la carpeta temporal de este cuaderno. Si quieres repetir una sección que lee esos archivos después del cierre, vuelve a ejecutar desde Preparación.

In [9]:
temporal.cleanup()

#### Nota Codex — Repaso

Una lectura fiable necesita conocer la fuente, declarar lo que no conviene inferir y comprobar algunos resultados. Continúa con [Pandas I-III — Ejercicios Complemento Codex](08_Pandas_I_III_Ejercicios_Complemento_Codex.ipynb).

#### Nota Codex — Límites del complemento

El diagnóstico de nulos, duplicados, frecuencias y estadísticos se estudia en **Pandas IV**.
La conversión y limpieza de columnas, imputación, recodificación y tratamiento de texto se estudian en **Pandas V**.
`concat` y `merge` se estudian en **Pandas VI**.
Los gráficos y su preparación están en [Visualización con Pandas](../../18_VISUALIZACION/02_PANDAS/01_Notebooks/04_Visualization_II_Pandas.ipynb). Aquí no se repiten esos temas.

#### Nota Codex — Documentación de consulta

[read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html), [indexación](https://pandas.pydata.org/docs/user_guide/indexing.html) y [datos faltantes](https://pandas.pydata.org/docs/user_guide/missing_data.html). La ejecución de estos materiales se verifica con Pandas 2.3.1; no se depende de la inferencia de texto de una versión para conservar códigos.